# 영화리뷰 텍스트 감성분석하기

- 다양한 방법으로 Text Classification 태스크를 성공적으로 구현하였다.  
&rarr; 3가지 이상의 모델이 성공적으로 시도됨
- gensim을 활용하여 자체학습된 혹은 사전학습된 임베딩 레이어를 분석하였다.  
&rarr; gensim의 유사단어 찾기를 활용하여 자체학습한 임베딩과 사전학습 임베딩을 비교 분석함
- 한국어 Word2Vec을 활용하여 가시적인 성능향상을 달성했다.  
&rarr; 네이버 영화리뷰 데이터 감성분석 정확도를 85% 이상 달성함

# 0. 라이브러리 import 및 세팅

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from gensim.models import KeyedVectors
# from gensim.models import keyedvectors
from konlpy.tag import Mecab

import pandas as pd
import numpy as np
import glob
from collections import Counter

import matplotlib.pyplot as plt

In [2]:
VOCAB_SIZE = 20000 # 어휘 사전 크기
WORD_VECTOR_DIM = 128 # 자체 학습 시 word embeddig vector 차원

In [3]:
SAVE_PATH = "./best_models"

BATCH_SIZE = 512
LR = 0.005
W2V_LR = 0.001
EPOCHS = 20

criterion = nn.BCELoss()

In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

cuda


In [5]:
def get_save_model_path(model_name):
    save_model_path = f"{SAVE_PATH}/{model_name}.pt"
    return save_model_path

# 1. 데이터 준비 및 전처리

In [6]:
# 불용어 목록 작성
tokenizer = Mecab()
stopwords = ['의','가','이','은','들','는','좀','잘','걍','과','도','를','으로','자','에','와','한','하다']

In [7]:
# 문장 1개를 활용할 딕셔너리와 함께 주면, 단어 인덱스 리스트 벡터로 변환
# 모든 문장은 <BOS>로 시작 가정
def get_encoded_sentence(sentence, word_to_idx):
    return [word_to_idx['<BOS>']]+[word_to_idx[word] if word in word_to_idx else word_to_idx['<UNK>'] for word in sentence.split()]

# 여러 개의 문장 리스트를 한꺼번에 단어 인덱스 리스트 벡터로 encode하는 함수
def get_encoded_sentences(sentences, word_to_idx):
    return [get_encoded_sentence(sentence, word_to_idx) for sentence in sentences]

# 숫자 벡터로 encode된 문장을 원래대로 decode하는 함수
def get_decoded_sentence(encoded_sentence, idx_to_word):
    return ' '.join(idx_to_word[idx] if idx in idx_to_word else '<UNK>' for idx in encoded_sentence[1:])  #[1:]를 통해 <BOS>를 제외

# 여러 개의 숫자 벡터로 encode된 문장을 한꺼번에 원래대로 decode하는 함수
def get_decoded_sentences(encoded_sentences, idx_to_word):
    return [get_decoded_sentence(encoded_sentence, idx_to_word) for encoded_sentence in encoded_sentences]

In [8]:
def load_data(train_data, test_data, num_words=10000):
    '''
    train_data: DataFrame 타입
    train_data: DataFrame 타입
    num_words: vocab size

    0. row 별 str 형식의 문장 단위 데이터를 불러옴
    1. 중복 제거
    2. 결측치 제거
    3. 형태소 단위로 토큰화
    4. 불용어 제거
    5. 등장 횟수 순서대로 인덱싱하여 word_to_idx 생성 (길이: num_words)
    6. word_to_idx를 이용해 x_train, y_test에 있는 단어를 인덱스로 변환
    7. x_train, y_train, x_test, y_test, word_to_idx 반환
    '''
    
    print("VOCAB_SIZE: ", num_words)
    train_data.drop_duplicates(subset=['document'], inplace=True)
    train_data = train_data.dropna(how='any')
    test_data.drop_duplicates(subset=['document'], inplace=True)
    test_data = test_data.dropna(how='any')

    x_train = []
    for sentence in train_data['document']:
        tmp_x = tokenizer.morphs(sentence)
        tmp_x = [word for word in tmp_x if not word in stopwords]
        x_train.append(tmp_x)
    
    x_test = []
    for sentence in test_data['document']:
        tmp_x = tokenizer.morphs(sentence)
        tmp_x = [word for word in tmp_x if not word in stopwords]
        x_test.append(tmp_x)
    
    words = np.concatenate(x_train).tolist() # 각 문장의 토큰들을 하나의 리스트에 합침
    counter = Counter(words) # 전체 텍스트에서 각 토큰이 얼마나 사용됐는지 계수
    counter = counter.most_common(num_words-4) # 많이 쓰인 것부터 정렬(특수 토큰 4개 들어갈 자리 빼고)
    vocab = ['<PAD>','<BOS>','<UNK>','<UNUSED>'] + [key for key, _ in counter] # 어휘 사전에 특수 토큰 추가
    word_to_idx = {word:idx for idx, word in enumerate(vocab)} # key: word, value: index 형태의 어휘 사전 생성

    # 텍스트 데이터 "문장"별로 각 토큰을 word_to_idx로 정수 인덱싱
    def wordlist_to_idxlist(word_list):
        return [word_to_idx[word] if word in word_to_idx else word_to_idx['<UNK>'] for word in word_list]

    x_train = list(map(wordlist_to_idxlist, x_train))
    x_test = list(map(wordlist_to_idxlist, x_test))

    return x_train, np.array(list(train_data['label'])), x_test, np.array(list(test_data['label'])), word_to_idx

In [9]:
def pad_sequences(data, maxlen):
    padded_data = []
    for sentence in data:
        if len(sentence) < maxlen:
            sentence += [0]*(maxlen-len(sentence)) # MAX_LEN보다 짧은 문장은 뒤에 <PAD> 토큰을 채우고,
        else:
            sentence = sentence[:maxlen] # MAX_LEN보다 긴 문장은 MAX_LEN만큼 자름
        padded_data.append(sentence)
    return np.array(padded_data)

## 1.1. 데이터 분석

### 1.1.1. 데이터 불러오기 및 확인

In [10]:
# 원본 .txt 파일을 dataframe 형식으로 불러오기
train_data = pd.read_table("./sentiment_data/ratings_train.txt")
test_data = pd.read_table("./sentiment_data/ratings_test.txt")

In [11]:
train_data.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [12]:
x_train, y_train, x_test, y_test, word_to_idx = load_data(train_data, test_data, num_words=VOCAB_SIZE)

VOCAB_SIZE:  20000


In [13]:
idx_to_word = {v: k for k,v in word_to_idx.items()} 

In [115]:
print(train_data.iloc[0, 1])
# print(word_to_idx)
# print(idx_to_word)

아 더빙.. 진짜 짜증나네요 목소리


### 1.1.2. 최대 문장 길이 설정

In [15]:
total_data_txt = list(x_train) + list(x_test)

num_tokens = [len(tokens) for tokens in total_data_txt]
num_tokens = np.array(num_tokens)

print('문장 길이 평균: ', np.mean(num_tokens))
print('문장 길이 최대: ', np.max(num_tokens))
print('문장 길이 표준편차: ', np.std(num_tokens))
print('문장 길이 중위수: ', np.median(num_tokens))

문장 길이 평균:  15.9710452085861
문장 길이 최대:  116
문장 길이 표준편차:  12.844097739499528
문장 길이 중위수:  12.0


In [16]:
# 전체 데이터 중 "문장 길이 평균 + 2*표준편차" 길이를 MAX_LEN(최대 문장 길이)로 지정해서 사용
max_tokens = np.mean(num_tokens) + 2*np.std(num_tokens)
MAX_LEN = int(max_tokens)
print('pad_sequences maxlen: ', MAX_LEN)
print(f"전체 문장의 {np.sum(num_tokens < max_tokens) / len(num_tokens) *100:.2f}%가 maxlen 설정값 이내에 포함됩니다.")

pad_sequences maxlen:  41
전체 문장의 93.43%가 maxlen 설정값 이내에 포함됩니다.


In [17]:
print(MAX_LEN)

41


In [18]:
x_train_encoded = pad_sequences(x_train, MAX_LEN)
x_test_encoded = pad_sequences(x_test, MAX_LEN)

In [19]:
# padding 확인
x_train_encoded[0]

array([ 32,  74, 939,   4,   4,  39, 229,  20,  33, 747,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0])

## 1.2. Data Loader 생성

### 1.2.1. validation set 구성

In [20]:
# test set 크기 만큼 training set에서 분리해서 validation set 생성
len_val = len(x_test)
print(len_val)

# x_val = x_train_tensor[:len_val]
x_val = x_train_encoded[:len_val]
y_val = y_train[:len_val]

# partial_x_train = x_train_tensor[len_val:]
partial_x_train = x_train_encoded[len_val:]
partial_y_train = y_train[len_val:]

print(partial_x_train.shape)
print(partial_y_train.shape)

49157
(97025, 41)
(97025,)


### 1.2.2. Data Loader 생성

In [21]:
partial_x_train_tensor = torch.tensor(partial_x_train, dtype=torch.long)
partial_y_train_tensor = torch.tensor(partial_y_train, dtype=torch.float)

x_val_tensor = torch.tensor(x_val, dtype=torch.long)
y_val_tensor = torch.tensor(y_val, dtype=torch.float)

x_test_tensor = torch.tensor(x_test_encoded, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.float)

In [22]:
trainset = TensorDataset(partial_x_train_tensor, partial_y_train_tensor)
validset = TensorDataset(x_val_tensor, y_val_tensor)
testset = TensorDataset(x_test_tensor, y_test_tensor)

trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)
validloader = DataLoader(validset, batch_size=BATCH_SIZE, shuffle=False)
testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False)

# 2. 모델 정의 

RNN, LSTM 정의 후 모델 인스턴스 생성 시 파라미터 값을 통해 stacked RNN, BiLSTM 모델 사용

## 2.1. RNN

In [23]:
class SentRNN(nn.Module):
    def __init__(self, vocab_size, word_vector_dim, n_layers=1, embedding_mat=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, word_vector_dim)
        if embedding_mat is not None: # 사전 학습 된 임베딩을 사용할 경우
            self.embedding.weight = nn.Parameter(torch.tensor(embedding_mat, dtype=torch.float32))
            self.embedding.weight.requires_grad = True
            print("using pre-trained word vectors...")
        self.dropout = nn.Dropout(0.5)
        # n_layers로 stacked rnn 조절
        self.rnn = nn.RNN(input_size=word_vector_dim, hidden_size=word_vector_dim, num_layers=n_layers, 
                           nonlinearity='tanh', batch_first=True)
        self.fc1 = nn.Linear(word_vector_dim, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.embedding(x)
        out, hh = self.rnn(x)
        out = out[:, -1, :]
        out = self.dropout(out)
        out = F.relu(self.fc1(out))
        out = torch.sigmoid(self.fc2(out))
        return out


## 2.2. LSTM

In [24]:
class SentLSTM(nn.Module):
    def __init__(self, vocab_size, word_vector_dim, bidirectional=False, embedding_mat=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, word_vector_dim)
        if embedding_mat is not None: # 사전 학습 된 임베딩을 사용할 경우
            self.embedding.weight = nn.Parameter(torch.tensor(embedding_mat, dtype=torch.float32))
            self.embedding.weight.requires_grad = True
            print("using pre-trained word vectors...")
        self.dropout = nn.Dropout(0.5)
        # bidirectional로 biLSTM 설정 -> output이 출력단, 입력단 2개가 나오므로 fc1의 input size를 2배
        self.lstm = nn.LSTM(word_vector_dim, word_vector_dim, batch_first=True, bidirectional=bidirectional)
        if bidirectional:
            self.fc1 = nn.Linear(word_vector_dim*2, 64)
        else:    
            self.fc1 = nn.Linear(word_vector_dim, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.embedding(x)
        out, (hh, cn) = self.lstm(x)
        if self.lstm.bidirectional:
            out = torch.mean(out, dim=1) # biLSTM 설정 시 양방향 정보 평균해서 사용
        else:
            out = out[:, -1, :]
        out = self.dropout(out)
        out = F.relu(self.fc1(out))        
        out = torch.sigmoid(self.fc2(out))
        return out


# 3. 모델 학습

## 3.1. 학습 및 테스트 함수 정의

In [25]:
def make_results_dict():
    res_dict = {
        'train_losses': [],
        'val_losses': [],
        'train_accs': [],
        'val_accs': []               
    }    
    return res_dict

In [26]:
def train_model(model, epochs, criterion, lr, trainloader, validloader, save_model_path):
    
    res_dict = make_results_dict()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    best_loss = 99999

    for epoch in range(epochs):
        model.train()
        r_loss = 0.
        correct = 0
        total = 0

        for inputs, labels in trainloader:        
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs.squeeze(), labels)
            loss.backward()
            optimizer.step()

            r_loss += loss.item() # 배치에 대한 평균 loss

            predicted = (outputs.squeeze() > 0.5).float()
            correct += (predicted == labels).sum().item()
            total += len(labels)
        
        res_dict['train_losses'].append(r_loss / len(trainloader))
        res_dict['train_accs'].append(correct / total)

        model.eval()
        val_loss = 0.
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in validloader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

                outputs = model(inputs)
                loss = criterion(outputs.squeeze(), labels)
                val_loss += loss.item()
                
                predicted = (outputs.squeeze() > 0.5).float()
                val_correct += (predicted == labels).sum().item()
                val_total += len(labels)

            if val_loss/len(validloader) < best_loss:
                best_loss = val_loss/len(validloader)
                # optimizer도 같이 save하면 이어서 학습 가능
                torch.save({"model": model.state_dict(),
                            "epoch": epoch+1,
                            "optimizer": optimizer}, save_model_path)
            
            res_dict['val_losses'].append(val_loss / len(validloader))
            res_dict['val_accs'].append(val_correct / val_total)
        
        print(f"Epoch {epoch+1}/{epochs} - " 
            f"Train Loss: {res_dict['train_losses'][-1]:.4f}, Train Accuracy: {res_dict['train_accs'][-1]*100:.2f}% - "
            f"Validation Loss {res_dict['val_losses'][-1]:.4f}, Validation Accuracy: {res_dict['val_accs'][-1]*100:.2f}%")
        
    
    print("Training has been finished...")
    print(f"Best validation loss: {best_loss:.4f}")
    return res_dict


In [27]:
def test_model(model, criterion, testloader):    
    model.eval()
    test_loss = 0.
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            outputs = model(inputs)

            loss = criterion(outputs.squeeze(), labels)
            
            test_loss += loss.item()
            predicted = (outputs.squeeze() > 0.5).float()
            test_correct += (predicted == labels).sum().item()
            test_total += len(labels)
    test_acc = test_correct/test_total

    print(f"Test Loss: {test_loss / len(testloader):.4f}, Test Accuracy: {test_acc*100:.2f}%")

    return test_acc


## 3.2. 모델 별 학습 및 결과 저장

In [28]:
test_acc_results = {}

### 3.2.1. RNN

In [29]:
# n_layers=1인 기본 RNN
model_sent_rnn = SentRNN(VOCAB_SIZE, WORD_VECTOR_DIM).to(DEVICE)
print(model_sent_rnn)

model_name = model_sent_rnn._get_name()
save_model_path = get_save_model_path(model_name)
print(save_model_path)

SentRNN(
  (embedding): Embedding(20000, 128)
  (dropout): Dropout(p=0.5, inplace=False)
  (rnn): RNN(128, 128, batch_first=True)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)
./best_models/SentRNN.pt


In [30]:
res_sent_rnn = train_model(model_sent_rnn, EPOCHS, criterion, LR, trainloader, validloader, save_model_path)

Epoch 1/20 - Train Loss: 0.6944, Train Accuracy: 50.33% - Validation Loss 0.6928, Validation Accuracy: 50.06%
Epoch 2/20 - Train Loss: 0.6931, Train Accuracy: 50.45% - Validation Loss 0.6926, Validation Accuracy: 50.75%
Epoch 3/20 - Train Loss: 0.6917, Train Accuracy: 50.56% - Validation Loss 0.6948, Validation Accuracy: 51.18%
Epoch 4/20 - Train Loss: 0.6898, Train Accuracy: 51.05% - Validation Loss 0.6913, Validation Accuracy: 50.32%
Epoch 5/20 - Train Loss: 0.6852, Train Accuracy: 51.31% - Validation Loss 0.6881, Validation Accuracy: 50.84%
Epoch 6/20 - Train Loss: 0.6796, Train Accuracy: 51.99% - Validation Loss 0.6870, Validation Accuracy: 51.31%
Epoch 7/20 - Train Loss: 0.6767, Train Accuracy: 52.08% - Validation Loss 0.6936, Validation Accuracy: 50.93%
Epoch 8/20 - Train Loss: 0.6690, Train Accuracy: 55.01% - Validation Loss 0.6907, Validation Accuracy: 51.36%
Epoch 9/20 - Train Loss: 0.6744, Train Accuracy: 52.78% - Validation Loss 0.6941, Validation Accuracy: 51.19%
Epoch 10/2

In [31]:
# 저장한 모델 불러오기
loaded_model = SentRNN(VOCAB_SIZE, WORD_VECTOR_DIM).to(DEVICE)
model_path = get_save_model_path(f"{loaded_model._get_name()}")
print(model_path)

ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
loaded_model.load_state_dict(ckpt['model'])

./best_models/SentRNN.pt


<All keys matched successfully>

In [32]:
test_res_sent_rnn = test_model(loaded_model, criterion, testloader) # 불러온 모델로 test acc 측정
test_acc_results[model_name] = test_res_sent_rnn # test acc 저장

Test Loss: 0.6209, Test Accuracy: 70.53%


### 3.2.2. Stacked RNN (n_layers=3)

In [33]:
# n_layers로 RNN cell stack
model_sent_stacked_rnn = SentRNN(VOCAB_SIZE, WORD_VECTOR_DIM, n_layers=3).to(DEVICE)
print(model_sent_stacked_rnn)

model_name = model_sent_stacked_rnn._get_name() + "_stacked"
save_model_path = get_save_model_path(model_name)
print(save_model_path)

SentRNN(
  (embedding): Embedding(20000, 128)
  (dropout): Dropout(p=0.5, inplace=False)
  (rnn): RNN(128, 128, num_layers=3, batch_first=True)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)
./best_models/SentRNN_stacked.pt


In [34]:
res_sent_stacked_rnn = train_model(model_sent_stacked_rnn, EPOCHS, criterion, LR, trainloader, validloader, save_model_path)

Epoch 1/20 - Train Loss: 0.6946, Train Accuracy: 50.01% - Validation Loss 0.6936, Validation Accuracy: 49.67%
Epoch 2/20 - Train Loss: 0.6935, Train Accuracy: 50.04% - Validation Loss 0.6944, Validation Accuracy: 49.66%
Epoch 3/20 - Train Loss: 0.6935, Train Accuracy: 49.91% - Validation Loss 0.6927, Validation Accuracy: 50.22%
Epoch 4/20 - Train Loss: 0.6939, Train Accuracy: 49.98% - Validation Loss 0.6927, Validation Accuracy: 50.45%
Epoch 5/20 - Train Loss: 0.6938, Train Accuracy: 50.24% - Validation Loss 0.6938, Validation Accuracy: 49.41%
Epoch 6/20 - Train Loss: 0.6940, Train Accuracy: 50.13% - Validation Loss 0.6931, Validation Accuracy: 50.08%
Epoch 7/20 - Train Loss: 0.6938, Train Accuracy: 49.95% - Validation Loss 0.6951, Validation Accuracy: 49.43%
Epoch 8/20 - Train Loss: 0.6939, Train Accuracy: 50.23% - Validation Loss 0.6935, Validation Accuracy: 50.08%
Epoch 9/20 - Train Loss: 0.6939, Train Accuracy: 49.82% - Validation Loss 0.6927, Validation Accuracy: 50.57%
Epoch 10/2

In [36]:
loaded_model = SentRNN(VOCAB_SIZE, WORD_VECTOR_DIM, n_layers=3).to(DEVICE)
model_path = get_save_model_path(f"{loaded_model._get_name()}"+"_stacked")
print(model_path)

ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
loaded_model.load_state_dict(ckpt['model'])

./best_models/SentRNN_stacked.pt


<All keys matched successfully>

In [37]:
test_res_sent_stacked = test_model(loaded_model, criterion, testloader) # 불러온 모델로 test acc 측정
test_acc_results[model_name] = test_res_sent_stacked # test acc 저장

Test Loss: 0.6930, Test Accuracy: 50.79%


### 3.2.3 LSTM

In [38]:
model_sent_lstm = SentLSTM(VOCAB_SIZE, WORD_VECTOR_DIM).to(DEVICE)
print(model_sent_lstm)

model_name = model_sent_lstm._get_name()
save_model_path = get_save_model_path(model_name)
print(save_model_path)

SentLSTM(
  (embedding): Embedding(20000, 128)
  (dropout): Dropout(p=0.5, inplace=False)
  (lstm): LSTM(128, 128, batch_first=True)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)
./best_models/SentLSTM.pt


In [39]:
res_sent_lstm = train_model(model_sent_lstm, EPOCHS, criterion, LR, trainloader, validloader, save_model_path)

Epoch 1/20 - Train Loss: 0.6284, Train Accuracy: 60.93% - Validation Loss 0.4371, Validation Accuracy: 81.14%
Epoch 2/20 - Train Loss: 0.3730, Train Accuracy: 84.23% - Validation Loss 0.3498, Validation Accuracy: 84.95%
Epoch 3/20 - Train Loss: 0.2842, Train Accuracy: 88.62% - Validation Loss 0.3379, Validation Accuracy: 85.54%
Epoch 4/20 - Train Loss: 0.2264, Train Accuracy: 91.26% - Validation Loss 0.3617, Validation Accuracy: 85.36%
Epoch 5/20 - Train Loss: 0.1806, Train Accuracy: 93.28% - Validation Loss 0.3836, Validation Accuracy: 85.32%
Epoch 6/20 - Train Loss: 0.1478, Train Accuracy: 94.66% - Validation Loss 0.4134, Validation Accuracy: 85.09%
Epoch 7/20 - Train Loss: 0.1195, Train Accuracy: 95.89% - Validation Loss 0.4416, Validation Accuracy: 84.95%
Epoch 8/20 - Train Loss: 0.1026, Train Accuracy: 96.54% - Validation Loss 0.5247, Validation Accuracy: 84.80%
Epoch 9/20 - Train Loss: 0.0904, Train Accuracy: 97.01% - Validation Loss 0.5000, Validation Accuracy: 84.47%
Epoch 10/2

In [40]:
# 저장한 모델 불러오기
loaded_model = SentLSTM(VOCAB_SIZE, WORD_VECTOR_DIM).to(DEVICE)
model_path = get_save_model_path(f"{loaded_model._get_name()}")
print(model_path)

ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
loaded_model.load_state_dict(ckpt['model'])

./best_models/SentLSTM.pt


<All keys matched successfully>

In [41]:
test_res_sent_lstm = test_model(loaded_model, criterion, testloader) # 불러온 모델로 test acc 측정
test_acc_results[model_name] = test_res_sent_lstm # test acc 저장

Test Loss: 0.3464, Test Accuracy: 84.99%


### 3.2.4. BiLSTM

In [42]:
# bidirectional=True로 biLSTM 설정
model_sent_bilstm = SentLSTM(VOCAB_SIZE, WORD_VECTOR_DIM, bidirectional=True).to(DEVICE)
print(model_sent_bilstm)

model_name = model_sent_bilstm._get_name()+"_bilstm"
save_model_path = get_save_model_path(model_name)
print(save_model_path)

SentLSTM(
  (embedding): Embedding(20000, 128)
  (dropout): Dropout(p=0.5, inplace=False)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (fc1): Linear(in_features=256, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)
./best_models/SentLSTM_bilstm.pt


In [43]:
res_sent_bilstm = train_model(model_sent_bilstm, EPOCHS, criterion, LR, trainloader, validloader, save_model_path)

Epoch 1/20 - Train Loss: 0.4421, Train Accuracy: 78.10% - Validation Loss 0.3376, Validation Accuracy: 85.20%
Epoch 2/20 - Train Loss: 0.2987, Train Accuracy: 87.53% - Validation Loss 0.3189, Validation Accuracy: 86.07%
Epoch 3/20 - Train Loss: 0.2347, Train Accuracy: 90.67% - Validation Loss 0.3289, Validation Accuracy: 86.22%
Epoch 4/20 - Train Loss: 0.1827, Train Accuracy: 92.95% - Validation Loss 0.3564, Validation Accuracy: 85.82%
Epoch 5/20 - Train Loss: 0.1388, Train Accuracy: 94.83% - Validation Loss 0.3996, Validation Accuracy: 85.58%
Epoch 6/20 - Train Loss: 0.1088, Train Accuracy: 95.99% - Validation Loss 0.4626, Validation Accuracy: 85.33%
Epoch 7/20 - Train Loss: 0.0875, Train Accuracy: 96.85% - Validation Loss 0.5079, Validation Accuracy: 85.25%
Epoch 8/20 - Train Loss: 0.0738, Train Accuracy: 97.35% - Validation Loss 0.5726, Validation Accuracy: 85.20%
Epoch 9/20 - Train Loss: 0.0634, Train Accuracy: 97.79% - Validation Loss 0.6358, Validation Accuracy: 85.04%
Epoch 10/2

In [44]:
loaded_model = SentLSTM(VOCAB_SIZE, WORD_VECTOR_DIM, bidirectional=True).to(DEVICE)
model_path = get_save_model_path(f"{loaded_model._get_name()}"+"_bilstm")
print(model_path)

loaded_model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=False)['model'])

./best_models/SentLSTM_bilstm.pt


<All keys matched successfully>

In [45]:
test_res_sent_bilstm = test_model(loaded_model, criterion, testloader) # 불러온 모델로 test acc 측정
test_acc_results[model_name] = test_res_sent_bilstm # test acc 저장

Test Loss: 0.3313, Test Accuracy: 85.54%


# 4. 한국어 Word2Vec 임베딩 활용

## 4.1. 한국어 임베딩 load

In [46]:
w2v_path = "./sentiment_data/word2vec_ko.model"
w2v = KeyedVectors.load(w2v_path)

In [47]:
# pre-trained word embedding 차원 확인
vec = w2v.wv['나라']
print(vec.shape)
print(w2v.wv.vector_size)

(100,)
100


In [48]:
vocab_size = VOCAB_SIZE
pretrained_word_vector_dim = w2v.wv.vector_size # 모델 정의 시 내부에서 필요한 WORD_VECTOR_DIM을 바꿔줘야함 
embedding_mat = np.random.rand(vocab_size, pretrained_word_vector_dim) # 모델에 넣어줄 사전 학습된 word embedding

for  i in range(4, vocab_size): # 특수 토큰 이후부터
    if idx_to_word[i] in w2v.wv:
        embedding_mat[i] = w2v.wv[idx_to_word[i]]

In [49]:
embedding_mat.shape

(20000, 100)

## 4.2. 한국어 임베딩 적용

### 4.2.1. RNN

In [50]:
model_sent_rnn_w2v = SentRNN(VOCAB_SIZE, pretrained_word_vector_dim, embedding_mat=embedding_mat).to(DEVICE)
print(model_sent_rnn_w2v)

model_name = model_sent_rnn_w2v._get_name()+"_w2v"
save_model_path = get_save_model_path(model_name)
print(save_model_path)

using pre-trained word vectors...
SentRNN(
  (embedding): Embedding(20000, 100)
  (dropout): Dropout(p=0.5, inplace=False)
  (rnn): RNN(100, 100, batch_first=True)
  (fc1): Linear(in_features=100, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)
./best_models/SentRNN_w2v.pt


In [51]:
res_sent_rnn_w2v = train_model(model_sent_rnn_w2v, EPOCHS, criterion, LR, trainloader, validloader, save_model_path)

Epoch 1/20 - Train Loss: 0.6935, Train Accuracy: 50.63% - Validation Loss 0.6919, Validation Accuracy: 50.74%
Epoch 2/20 - Train Loss: 0.6919, Train Accuracy: 50.58% - Validation Loss 0.6911, Validation Accuracy: 51.06%
Epoch 3/20 - Train Loss: 0.6852, Train Accuracy: 53.70% - Validation Loss 0.6841, Validation Accuracy: 54.08%
Epoch 4/20 - Train Loss: 0.6781, Train Accuracy: 55.97% - Validation Loss 0.6828, Validation Accuracy: 54.11%
Epoch 5/20 - Train Loss: 0.6793, Train Accuracy: 55.48% - Validation Loss 0.6801, Validation Accuracy: 56.96%
Epoch 6/20 - Train Loss: 0.6671, Train Accuracy: 59.73% - Validation Loss 0.6805, Validation Accuracy: 57.53%
Epoch 7/20 - Train Loss: 0.6629, Train Accuracy: 60.42% - Validation Loss 0.6655, Validation Accuracy: 59.95%
Epoch 8/20 - Train Loss: 0.6659, Train Accuracy: 58.30% - Validation Loss 0.6679, Validation Accuracy: 60.90%
Epoch 9/20 - Train Loss: 0.6681, Train Accuracy: 57.65% - Validation Loss 0.6937, Validation Accuracy: 50.88%
Epoch 10/2

In [52]:
# 저장한 모델 불러오기
loaded_model = SentRNN(VOCAB_SIZE, pretrained_word_vector_dim, embedding_mat=embedding_mat).to(DEVICE)
model_path = get_save_model_path(f"{loaded_model._get_name()}_w2v")
print(model_path)

ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
loaded_model.load_state_dict(ckpt['model'])

using pre-trained word vectors...
./best_models/SentRNN_w2v.pt


<All keys matched successfully>

In [53]:
test_res_sent_rnn_w2v = test_model(loaded_model, criterion, testloader) # 불러온 모델로 test acc 측정
test_acc_results[model_name] = test_res_sent_rnn_w2v # test acc 저장

Test Loss: 0.5893, Test Accuracy: 72.63%


### 4.2.2. Stacked RNN (n_layers=3)

In [54]:
model_sent_rnn_stacked_w2v = SentRNN(VOCAB_SIZE, pretrained_word_vector_dim, n_layers=3, embedding_mat=embedding_mat).to(DEVICE)
print(model_sent_rnn_w2v)

model_name = model_sent_stacked_rnn._get_name()+"_stacked_w2v"
save_model_path = get_save_model_path(model_name)
print(save_model_path)

using pre-trained word vectors...
SentRNN(
  (embedding): Embedding(20000, 100)
  (dropout): Dropout(p=0.5, inplace=False)
  (rnn): RNN(100, 100, batch_first=True)
  (fc1): Linear(in_features=100, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)
./best_models/SentRNN_stacked_w2v.pt


In [55]:
res_sent_rnn_stacked_w2v = train_model(model_sent_rnn_stacked_w2v, EPOCHS, criterion, LR, trainloader, validloader, save_model_path)

Epoch 1/20 - Train Loss: 0.6939, Train Accuracy: 50.38% - Validation Loss 0.6932, Validation Accuracy: 50.08%
Epoch 2/20 - Train Loss: 0.6930, Train Accuracy: 50.30% - Validation Loss 0.6930, Validation Accuracy: 50.08%
Epoch 3/20 - Train Loss: 0.6929, Train Accuracy: 50.28% - Validation Loss 0.6940, Validation Accuracy: 50.56%
Epoch 4/20 - Train Loss: 0.6926, Train Accuracy: 50.43% - Validation Loss 0.6930, Validation Accuracy: 50.50%
Epoch 5/20 - Train Loss: 0.6924, Train Accuracy: 50.31% - Validation Loss 0.6928, Validation Accuracy: 50.06%
Epoch 6/20 - Train Loss: 0.6904, Train Accuracy: 50.79% - Validation Loss 0.6894, Validation Accuracy: 51.09%
Epoch 7/20 - Train Loss: 0.6900, Train Accuracy: 51.03% - Validation Loss 0.6906, Validation Accuracy: 50.06%
Epoch 8/20 - Train Loss: 0.6885, Train Accuracy: 51.11% - Validation Loss 0.6946, Validation Accuracy: 51.41%
Epoch 9/20 - Train Loss: 0.6866, Train Accuracy: 51.51% - Validation Loss 0.6894, Validation Accuracy: 50.93%
Epoch 10/2

In [56]:
# 저장한 모델 불러오기
loaded_model = SentRNN(VOCAB_SIZE, pretrained_word_vector_dim, n_layers=3, embedding_mat=embedding_mat).to(DEVICE)
model_path = get_save_model_path(f"{loaded_model._get_name()}_stacked_w2v")
print(model_path)

ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
loaded_model.load_state_dict(ckpt['model'])

using pre-trained word vectors...
./best_models/SentRNN_stacked_w2v.pt


<All keys matched successfully>

In [57]:
test_res_sent_rnn_stacked_w2v = test_model(loaded_model, criterion, testloader) # 불러온 모델로 test acc 측정
test_acc_results[model_name] = test_res_sent_rnn_stacked_w2v # test acc 저장

Test Loss: 0.6439, Test Accuracy: 66.87%


### 4.2.3. LSTM

In [60]:
model_sent_lstm_w2v = SentLSTM(VOCAB_SIZE, pretrained_word_vector_dim, embedding_mat=embedding_mat).to(DEVICE)
print(model_sent_lstm_w2v)

model_name = model_sent_lstm_w2v._get_name() + "_w2v"
save_model_path = get_save_model_path(model_name)
print(save_model_path)

using pre-trained word vectors...
SentLSTM(
  (embedding): Embedding(20000, 100)
  (dropout): Dropout(p=0.5, inplace=False)
  (lstm): LSTM(100, 100, batch_first=True)
  (fc1): Linear(in_features=100, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)
./best_models/SentLSTM_w2v.pt


In [61]:
res_sent_lstm_w2v = train_model(model_sent_lstm_w2v, EPOCHS, criterion, LR, trainloader, validloader, save_model_path)

Epoch 1/20 - Train Loss: 0.6919, Train Accuracy: 50.63% - Validation Loss 0.6861, Validation Accuracy: 51.18%
Epoch 2/20 - Train Loss: 0.4726, Train Accuracy: 76.73% - Validation Loss 0.3627, Validation Accuracy: 83.92%
Epoch 3/20 - Train Loss: 0.3227, Train Accuracy: 86.64% - Validation Loss 0.3493, Validation Accuracy: 84.97%
Epoch 4/20 - Train Loss: 0.2686, Train Accuracy: 89.18% - Validation Loss 0.3517, Validation Accuracy: 84.98%
Epoch 5/20 - Train Loss: 0.2298, Train Accuracy: 91.04% - Validation Loss 0.3820, Validation Accuracy: 85.14%
Epoch 6/20 - Train Loss: 0.1993, Train Accuracy: 92.41% - Validation Loss 0.3869, Validation Accuracy: 85.13%
Epoch 7/20 - Train Loss: 0.1730, Train Accuracy: 93.59% - Validation Loss 0.4074, Validation Accuracy: 84.75%
Epoch 8/20 - Train Loss: 0.1541, Train Accuracy: 94.29% - Validation Loss 0.4531, Validation Accuracy: 84.65%
Epoch 9/20 - Train Loss: 0.1361, Train Accuracy: 95.15% - Validation Loss 0.4494, Validation Accuracy: 84.60%
Epoch 10/2

In [62]:
# 저장한 모델 불러오기
loaded_model = SentLSTM(VOCAB_SIZE, pretrained_word_vector_dim, embedding_mat=embedding_mat).to(DEVICE)
model_path = get_save_model_path(f"{loaded_model._get_name()}_w2v")
print(model_path)

ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
loaded_model.load_state_dict(ckpt['model'])

using pre-trained word vectors...
./best_models/SentLSTM_w2v.pt


<All keys matched successfully>

In [63]:
test_res_sent_lstm_w2v = test_model(loaded_model, criterion, testloader) # 불러온 모델로 test acc 측정
test_acc_results[model_name] = test_res_sent_lstm_w2v # test acc 저장

Test Loss: 0.3604, Test Accuracy: 84.71%


### 4.2.4. BiLSTM

In [64]:
model_sent_bilstm_w2v = SentLSTM(VOCAB_SIZE, pretrained_word_vector_dim, bidirectional=True, embedding_mat=embedding_mat).to(DEVICE)
print(model_sent_bilstm_w2v)

model_name = model_sent_lstm_w2v._get_name()+"_bilstm_w2v"
save_model_path = get_save_model_path(model_name)
print(save_model_path)

using pre-trained word vectors...
SentLSTM(
  (embedding): Embedding(20000, 100)
  (dropout): Dropout(p=0.5, inplace=False)
  (lstm): LSTM(100, 100, batch_first=True, bidirectional=True)
  (fc1): Linear(in_features=200, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)
./best_models/SentLSTM_bilstm_w2v.pt


In [65]:
res_sent_bilstm_w2v = train_model(model_sent_bilstm_w2v, EPOCHS, criterion, LR, trainloader, validloader, save_model_path)

Epoch 1/20 - Train Loss: 0.4440, Train Accuracy: 78.46% - Validation Loss 0.3386, Validation Accuracy: 85.27%
Epoch 2/20 - Train Loss: 0.3048, Train Accuracy: 87.12% - Validation Loss 0.3204, Validation Accuracy: 86.17%
Epoch 3/20 - Train Loss: 0.2539, Train Accuracy: 89.68% - Validation Loss 0.3198, Validation Accuracy: 86.21%
Epoch 4/20 - Train Loss: 0.2183, Train Accuracy: 91.37% - Validation Loss 0.3399, Validation Accuracy: 85.88%
Epoch 5/20 - Train Loss: 0.1865, Train Accuracy: 92.68% - Validation Loss 0.3693, Validation Accuracy: 85.69%
Epoch 6/20 - Train Loss: 0.1607, Train Accuracy: 93.87% - Validation Loss 0.4381, Validation Accuracy: 85.39%
Epoch 7/20 - Train Loss: 0.1400, Train Accuracy: 94.73% - Validation Loss 0.4647, Validation Accuracy: 85.09%
Epoch 8/20 - Train Loss: 0.1224, Train Accuracy: 95.50% - Validation Loss 0.4886, Validation Accuracy: 84.97%
Epoch 9/20 - Train Loss: 0.1100, Train Accuracy: 95.89% - Validation Loss 0.5166, Validation Accuracy: 84.93%
Epoch 10/2

In [66]:
# 저장한 모델 불러오기
loaded_model = SentLSTM(VOCAB_SIZE, pretrained_word_vector_dim, bidirectional=True, embedding_mat=embedding_mat).to(DEVICE)
model_path = get_save_model_path(f"{loaded_model._get_name()}_bilstm_w2v")
print(model_path)

ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
loaded_model.load_state_dict(ckpt['model'])

using pre-trained word vectors...
./best_models/SentLSTM_bilstm_w2v.pt


<All keys matched successfully>

In [67]:
test_res_sent_bilstm_w2v = test_model(loaded_model, criterion, testloader) # 불러온 모델로 test acc 측정
test_acc_results[model_name] = test_res_sent_bilstm_w2v # test acc 저장

Test Loss: 0.3337, Test Accuracy: 86.04%


# 5. 자체학습 임베딩 vs 사전학습 임베딩

## 5.1. 정량적 비교 (Test Accuracy)

In [70]:
test_acc_df = pd.DataFrame(data = {'model': list(test_acc_results.keys()), 
                                   'acc': list(test_acc_results.values())})

In [71]:
test_acc_df

,model,acc
0,SentRNN,0.705332
1,SentRNN_stacked,0.507944
2,SentLSTM,0.849930
3,SentLSTM_bilstm,0.855361
4,SentRNN_w2v,0.726306
5,SentRNN_stacked_w2v,0.668735
6,SentLSTM_w2v,0.847082
7,SentLSTM_bilstm_w2v,0.860366


- Test Accuracy 기준으로, BiLSTM 구조에 사전 학습된 임베딩을 적용한 경우 86.04%로 가장 성능이 높게 나옴
- Rnn의 경우 LSTM에 비해 성능이 떨어지는 것을 볼 수 있음  
&rarr; 출력층에서 멀수록 입력 데이터의 정보가 잊혀지는 RNN 구조의 특성을 생각하면, 전처리 시 문장 뒷부분에 padding 처리했기 때문에 모델이 출력층 부근에서 의미 있는 정보를 얻기 힘들었던 것으로 보임
- 또한, RNN cell을 여러 층으로 쌓았을 때 오히려 성능이 더 낮음을 확인함
- LSTM의 경우, 양방향 정보를 다 활용한 LSTM이 자체 학습과 사전 학습 임베딩 모두 일반 LSTM보다 성능이 더 좋은 것을 볼 수 있음

## 5.2. 정성적 비교 (임베딩)

In [102]:
import glob
import os

In [94]:
WORD = "영화"

In [101]:
MODEL_PATH = glob.glob(SAVE_PATH + '/*.pt')
print(MODEL_PATH)

['./best_models/SentLSTM.pt', './best_models/SentLSTM_w2v.pt', './best_models/SentRNN_stacked.pt', './best_models/SentRNN.pt', './best_models/SentRNN_w2v.pt', './best_models/SentLSTM_bilstm_w2v.pt', './best_models/SentRNN_stacked_w2v.pt', './best_models/SentLSTM_bilstm.pt']


In [ ]:
# 위에서 학습한 모델의 임베딩을 가져와서 특정 단어와 유사한 단어를 출력하는 함수
def most_similar(model_path, word, word_vector_dim, idx_to_word):
    
    # 1. 모델의 임베딩 가중치 가져오기 (CPU 텐서 -> numpy 변환)
    emb_weights = torch.load(model_path, map_location=DEVICE, weights_only=False)['model']['embedding.weight'].detach().cpu().numpy()
    
    # 2. Gensim의 KeyedVectors 객체 생성
    kv = KeyedVectors(vector_size=word_vector_dim)

    # 3. 단어와 벡터 주입
    words = [idx_to_word[i] for i in range(len(idx_to_word))]
    kv.add_vectors(words, emb_weights)

    # 4. most_similar 사용 가능
    print(kv.most_similar(word))

자체 학습 임베딩(+ fine tuning: *_w2v.pt)으로 유사 단어 검색 결과

In [110]:
for model_name in os.listdir("./best_models/"):
    if 'w2v' in model_name:
        print(model_name, end=": ")
        most_similar(SAVE_PATH+'/' + model_name, WORD, pretrained_word_vector_dim, idx_to_word)
    else:
        print(model_name, end=": ")
        most_similar(SAVE_PATH+'/' + model_name, WORD, WORD_VECTOR_DIM, idx_to_word)

SentLSTM.pt: [('얘긴', 0.3329887390136719), ('바다', 0.32091981172561646), ('프리먼', 0.3144347369670868), ('달린다', 0.3037763833999634), ('주저리', 0.3011007010936737), ('오연서', 0.29522350430488586), ('14', 0.2951848804950714), ('흘러간다', 0.2942958474159241), ('별명', 0.2920803427696228), ('매춘부', 0.2901167571544647)]
SentLSTM_w2v.pt: [('드라마', 0.8238096833229065), ('뮤지컬', 0.7739382982254028), ('헐리우드', 0.7471902966499329), ('코미디', 0.7464884519577026), ('다큐멘터리', 0.7433651685714722), ('장편', 0.7275603413581848), ('애니메이션', 0.7185887098312378), ('로맨틱', 0.7157994508743286), ('소설', 0.7107517719268799), ('극영화', 0.7040461897850037)]
SentRNN_stacked.pt: [('사기', 0.3348276913166046), ('짱꼴라', 0.32779473066329956), ('일행', 0.32392746210098267), ('층', 0.3156897723674774), ('붙일', 0.31568634510040283), ('긴장감', 0.31521308422088623), ('직행', 0.29209044575691223), ('전진', 0.28479477763175964), ('미련없이', 0.2837505042552948), ('신이', 0.280105859041214)]
SentRNN.pt: [('이프', 0.36045458912849426), ('질러', 0.3326805830001831), ('안다고',

사전 학습 임베딩으로 유사 단어 검색 결과

In [113]:
w2v_path = "./sentiment_data/word2vec_ko.model"
w2v = KeyedVectors.load(w2v_path)

In [114]:
print(w2v.wv.most_similar(WORD))

[('드라마', 0.8418774008750916), ('뮤지컬', 0.7775140404701233), ('코미디', 0.7489107251167297), ('다큐멘터리', 0.7401294708251953), ('헐리우드', 0.7397844195365906), ('애니메이션', 0.7170552015304565), ('독립영화', 0.7113528251647949), ('로맨틱', 0.7107657194137573), ('장편', 0.7101576924324036), ('극영화', 0.7045413255691528)]


- 사전 학습 임베딩을 이용해 "영화"라는 단어와 유사한 단어를 검색했을 때, "드라마", "뮤지컬"과 같은 단어가 유사도가 높게 나와 자체 학습한 경우보다 embedding vector가 의미적 유사도를 더 잘 포착한 상태임을 확인할 수 있음

# 회고

- 네이버 영화리뷰 데이터에 대해, RNN과 LSTM모델로 자체학습 및 사전학습 임베딩을 이용해 감성분석 task를 수행하였음
- 자체학습 시 초반부터 생각보다 성능이 잘 나온 점과, 사전학습한 임베딩을 사용했을 때 stackedRNN을 제외하고는 성능이 드라마틱하게 오르지 않은 점이 의외였음